# Lab 5 — Part 1: Azure readiness
**6688012 · ITCS355 · Azure ต่อจาก Lab 4**

เปิดใน VS Code ผ่าน **WSL** แล้วเลือก kernel ของ environment `itcs355-lab3` เดิม
รัน code cell ด้านล่างเพียงครั้งเดียว แล้วส่งข้อความตั้งแต่ `SEND THIS SUMMARY` กลับมา

ตรวจ environment, อ่าน identifiers จาก `cloud.env` เดิม, ตรวจ Azure ML/โมเดล/ทรัพยากร/โควตา และดึงราคา VM ปัจจุบัน
เขียนผลลง `lab5/reports/preflight.json` ไม่สร้าง resource ไม่ฝึกโมเดล ไม่ install และไม่ commit/push
รันซ้ำจะเก็บรายงานเดิมไว้ใน `preflight-history/`

**ยังไม่ใช่ managed pipeline ของ Task 1**: ต้องตรวจผลและ commit estimate ก่อนเริ่มงานที่คิดเงิน
quota/ราคาเป็นข้อมูลวางแผน ไม่ใช่การรับรอง capacity หรือยอด billing

ถ้า Azure login หมดอายุให้รัน `az login` ใน WSL terminal เองแล้วรัน cell ใหม่
ถ้าไม่มี progress ใหม่เกิน 5 นาทีให้กด Interrupt และส่ง output ล่าสุด ไม่เปิด cell ซ้อนกัน


In [1]:
from pathlib import Path
import platform
import subprocess
import sys

if platform.system() != 'Linux' or 'microsoft' not in platform.release().lower():
    raise RuntimeError('เลือก Python kernel ฝั่ง WSL ของ environment itcs355-lab3 ก่อนรันครับ')

here = Path.cwd().resolve()
lab = next((base / 'lab5' for base in (here, *here.parents)
            if (base / 'lab4' / 'README.md').is_file()
            and (base / 'lab5' / 'scripts' / 'preflight.py').is_file()), None)
if lab is None:
    raise RuntimeError('เปิด notebook จากโฟลเดอร์ 6688012-ITCS355-MLOP ผ่าน WSL ก่อนครับ')

print(f'Environment: {Path(sys.prefix).name} | Python {platform.python_version()}', flush=True)
print('Part 1: read-only Azure checks; no compute/endpoint will be created.', flush=True)
with subprocess.Popen(
    [sys.executable, '-u', '-m', 'scripts.preflight'], cwd=lab,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
) as task:
    try:
        for line in task.stdout:
            print(line, end='', flush=True)
        exit_code = task.wait()
    except KeyboardInterrupt:
        task.terminate()
        try:
            task.wait(timeout=10)
        except subprocess.TimeoutExpired:
            task.kill()
        raise

if exit_code:
    raise RuntimeError('Part 1 ยังไม่ผ่าน: ส่ง summary/error ด้านบนให้ผู้ช่วยตรวจ ไม่ต้อง install ใหม่ทั้งชุดครับ')
print('Part 1 เสร็จแล้ว — หยุดตรงนี้และส่ง summary กลับมาเพื่อตรวจขั้นถัดไปครับ', flush=True)


Environment: itcs355-lab3 | Python 3.11.16
Part 1: read-only Azure checks; no compute/endpoint will be created.
[13:14:22] 1/4 Check WSL, Python, tools and existing source
[13:14:27] 2/4 Read identifier-only config from previous Labs
[13:14:28] 3/4 Inspect Azure workspace/model, compute, endpoints, schedules and quota
[13:14:32] Read workspace and exact incumbent model
[13:14:39] Read compute
[13:14:39] Read endpoints
[13:14:41] Read schedules
[13:14:43] Read quota
[13:14:44] Read resource_inventory
[13:14:46] 4/4 Read current public DS2 v2 retail price meters

--- SEND THIS SUMMARY ---
{
  "status": "ready_for_planning",
  "local": {
    "wsl": true,
    "python": "3.11.16",
    "environment": "itcs355-lab3"
  },
  "packages": {
    "azure-ai-ml": "1.35.0",
    "azure-identity": "1.25.3",
    "ipykernel": "7.4.0"
  },
  "docker": {
    "reachable": true,
    "platform": "linux/x86_64"
  },
  "workspace": {
    "name": "itcs355-6688012",
    "location": "centralindia"
  },
  "incumbent

## หลังรัน
ส่ง summary ใน output กลับมา ไม่ต้องส่ง `cloud.env`, token หรือ key
`ready_for_planning` หมายถึงอ่าน workspace/model ได้ ยังไม่แปลว่ามีสิทธิ์ deploy หรือมีงบเพียงพอ

ขั้นถัดไปใช้ผลจริงเลือก compute/identity/image, เตรียม estimate และ notebook managed pipeline
ยังไม่ได้สร้าง notebook ของขั้น cloud ถัดไป เพื่อไม่ผูกกับ SKU/สิทธิ์ที่ยังไม่ตรวจ
Azure เป็น provider หลัก; second provider ตาม Task 4 และ LLM ตาม Part B ยังต้องเตรียมภายหลัง
